# Dataset Analysis

This notebook loads data and prepares image dataset for analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import os
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import os

image = cv2.imread(r"C:\Users\DELL\Documents\ImageDataset\Normales.jpg")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# threshold to separate beans from white background
_, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

# remove noise
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

# find contours
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

os.makedirs("beans", exist_ok=True)

i = 0
for c in contours:

    x,y,w,h = cv2.boundingRect(c)

    if w > 40 and h > 40:  # ignore noise
        crop = image[y:y+h, x:x+w]

        cv2.imwrite(f"beans/bean_{i}.jpg", crop)
        i += 1

print("Total beans detected:", i)

In [ ]:
import cv2
import numpy as np
import os

# Load image
img_path = r"C:\Users\DELL\Documents\ImageDataset\Concha.jpg"
img = cv2.imread(img_path)
if img is None:
    print("Cannot load image - check path")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

# Adaptive threshold (usually works best for beans)
thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 11, 3)

kernel = np.ones((3,3), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

os.makedirs("cropped_beans", exist_ok=True)

count = 0
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 300:          # skip very small noise
        continue

    x, y, w, h = cv2.boundingRect(cnt)

    # Bigger padding to avoid cutting beans
    pad = 60                # main fix - increase if still cut

    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img.shape[1], x + w + pad)
    y2 = min(img.shape[0], y + h + pad)

    # Skip if crop is too small
    if (x2 - x1) < 60 or (y2 - y1) < 60:
        continue

    crop = img[y1:y2, x1:x2]
    count += 1
    cv2.imwrite(f"cropped_beans/Concha_{count}.jpg", crop)

print(f"Saved {count} beans")

In [ ]:
import cv2
import numpy as np
import os

img_path = r"C:\Users\DELL\Documents\coffeeImageDataset\Normales.jpg"
img = cv2.imread(img_path)
if img is None:
    print("Cannot load image - check path")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 21, 4)

kernel = np.ones((3,3), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

cv2.imwrite(r"C:\Users\DELL\Documents\coffeeImageDataset\debug_thresh9.jpg", thresh)  # ← check this

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

output_dir = r"C:\Users\DELL\Documents\coffeeImageDataset\cropped"
os.makedirs(output_dir, exist_ok=True)

count = 0
TARGET_SIZE = (224, 224)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 470:          # lowered significantly
        continue

    x, y, w, h = cv2.boundingRect(cnt)

    pad = 65
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img.shape[1], x + w + pad)
    y2 = min(img.shape[0], y + h + pad)

    if (x2 - x1) < 60 or (y2 - y1) < 60:
        continue

    crop = img[y1:y2, x1:x2]
    crop_resized = cv2.resize(crop, TARGET_SIZE, interpolation=cv2.INTER_AREA)

    count += 1
    cv2.imwrite(f"{output_dir}\\Concha_{count}.jpg", crop_resized)

print(f"Saved {count} beans")

In [ ]:
# Visualize sample images from each class
def visualize_dataset(dataset_path, classes, samples_per_class=3):
    fig, axes = plt.subplots(len(classes), samples_per_class,
                             figsize=(15, 5*len(classes)))

    for i, class_name in enumerate(classes):
        class_path = os.path.join(dataset_path, class_name)
        images = os.listdir(class_path)[:samples_per_class]

        for j, img_name in enumerate(images):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{class_name}\n{img_name}")
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# Call the function
visualize_dataset(dataset_path, classes)

In [ ]:
import cv2
import numpy as np
import os

input_folder  = r"C:\Users\DELL\Documents\CBD_Coffee Bean Dataset\PB-II"
output_folder = r"C:\Users\DELL\Documents\CBD Dataset cropped\PB-II"
os.makedirs(output_folder, exist_ok=True)

count_global = 1

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(input_folder, filename)
    img = cv2.imread(img_path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 750:  # very low to force detection
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        pad = 50
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img.shape[1], x + w + pad)
        y2 = min(img.shape[0], y + h + pad)

        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_resized = cv2.resize(crop, (224, 224))

        out_name = f"A_{count_global}.jpg"
        out_path = os.path.join(output_folder, out_name)
        cv2.imwrite(out_path, crop_resized)

        count_global += 1

print("Finished. Check:", output_folder)

In [ ]:
import cv2
import numpy as np
import os

input_folder  = r"C:\Users\DELL\Documents\group A and B coffee\CGA\images"
output_folder = r"C:\Users\DELL\Documents\CBD Dataset\cropped_tight"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    path = os.path.join(input_folder, filename)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    # Try both directions — pick the one that works better for your images
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # or inverse: _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        continue

    # Take the largest contour (main bean)
    cnt = max(contours, key=cv2.contourArea)

    x, y, w, h = cv2.boundingRect(cnt)

    # Very tight crop + small safety margin
    margin = 8
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(img.shape[1], x + w + margin)
    y2 = min(img.shape[0], y + h + margin)

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        continue

    # Optional: resize to fixed size
    crop = cv2.resize(crop, (224, 224))

    out_path = os.path.join(output_folder, filename)
    cv2.imwrite(out_path, crop)

print("Done. Check:", output_folder)

### web scraping using python

In [18]:
import requests
from bs4 import BeautifulSoup

url = "https://www.geeksforgeeks.org/dsa/dsa-tutorial-learn-data-structures-and-algorithms/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

# Print all <p> tags
paragraphs = soup.find_all('p')
for p in paragraphs:
    print(p.get_text(strip=True))

In [13]:
from bs4 import BeautifulSoup
import requests

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

# Try CSS selector (handles multiple classes)
content = soup.select_one("div.a-wrapper")

if content:
    for par in content.find_all('p'):
        print(f"This is the select this class only p tages: {par.get_text(strip=True)}")
else:
    print("No article found")
print("----------------------------------")
print("App exists:", soup.find('div', class_='a-wrapper'))
print("Total p tags:", len(soup.find_all('p')))
print("----------------------------------")

None
App exists: None
Total p tags: 0


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin
from tqdm import tqdm

# ======================================================
# CONFIG
# ======================================================

URL = "https://pixabay.com/images/search/green%20coffee%20bean/"
SAVE_DIR = "green_coffee_images"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

os.makedirs(SAVE_DIR, exist_ok=True)


# ======================================================
# STEP 1: GET HTML PAGE
# ======================================================

response = requests.get(URL, headers=HEADERS)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


# ======================================================
# STEP 2: EXTRACT IMAGE LINKS
# ======================================================

images = soup.select("img")

image_urls = []

for img in images:
    src = img.get("src")
    if src and "https://cdn.pixabay.com" in src:
        image_urls.append(src)

print("Found images:", len(image_urls))


# ======================================================
# STEP 3: DOWNLOAD IMAGES
# ======================================================

for i, img_url in enumerate(tqdm(image_urls)):

    try:
        img_data = requests.get(img_url, headers=HEADERS).content

        file_path = os.path.join(SAVE_DIR, f"coffee_{i}.jpg")

        with open(file_path, "wb") as f:
            f.write(img_data)

    except Exception as e:
        print("Failed:", img_url)

print("Download finished")